# Modulo 1 — Analisis de Tickets (ETL & EDA)
**Objetivo:** Ingesta, limpieza y transformacion del dataset operativo de tickets para habilitar el descubrimiento de insights.

**Stack:** Pandas | Plotly | Python 3.14

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

file_path = 'data/Replica exacta de detalle dia - All tickets (15 días).csv'
df = pd.read_csv(file_path)

# Inspección inicial
print(f"Shape del dataset: {df.shape}")
print("\nTipos de datos originales:")
print(df.dtypes.head(15))
print("\nValores nulos por columna (Top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))


Shape del dataset: (672, 67)

Tipos de datos originales:
Fecha               object
Intervalo           object
Forecast Ordenes     int64
Ordenes              int64
% Desv Ordenes      object
Dif Ordenes          int64
Cancelados           int64
% SS Cancelados     object
Forecast % CR       object
Contact Rate        object
% Desv. CR          object
Forecast             int64
Inflow               int64
% Desvi. Inglow     object
Diff. Inflow         int64
dtype: object

Valores nulos por columna (Top 10):
Enc. Dsat Full Automated Comp           672
% Dsat full Automated Comp              672
Enc. Contestadas full_automated comp    672
% Snoozed CU                            150
Q Snoozed CU                            150
Enc. Neutros Comp                       107
% Reopen                                 72
Q_Reopen                                 72
Enc. Positivas Comp                       5
%_Auto_Controlados                        2
dtype: int64


## Normalizacion y Control de Calidad
Resolucion de inconsistencias en el formato de datos ingresado (formatos europeos, porcentajes string).
Estandarizacion vectorizada de la variable temporal a tipo `datetime` para integracion downstream.

In [2]:
def clean_european_numeric(series):
    '''
    Convierte series de pandas con formato europeo ("-13,96 %" o "1.234,56") a floats numéricos.
    '''
    # Si la serie ya es numérica, no hacemos nada
    if series.dtype != 'object':
        return series

    s = series.astype(str).str.strip()

    # Identificar filas que representan un porcentaje
    is_percent = s.str.contains('%', na=False)

    # Quitar el símbolo de porcentaje y espacios extra
    s = s.str.replace('%', '', regex=False).str.strip()

    # Si la serie contiene comas, asumimos formato europeo (coma = decimal, punto = miles)
    has_comma = s.str.contains(',', na=False).any()
    if has_comma:
        s = s.str.replace('.', '', regex=False) # quitar separador de miles
        s = s.str.replace(',', '.', regex=False) # coma decimal a punto decimal

    # Convertimos a float, forzando a NaN los valores no parseables ('coerce')
    s = pd.to_numeric(s, errors='coerce')

    # Los porcentajes se dividen por 100 para estandarizarlos entre 0 y 1
    s = np.where(is_percent & s.notna(), s / 100.0, s)
    return s

cols_to_skip = ['Fecha', 'Intervalo']
for col in df.columns:
    if col not in cols_to_skip:
        df[col] = clean_european_numeric(df[col])

# Conversión de Fechas:
df['Fecha_dt'] = pd.to_datetime(df['Fecha'], errors='coerce')
df['Hora_str'] = pd.to_datetime(df['Intervalo'], errors='coerce').dt.time.astype(str)

# Construimos el Timestamp final uniendo Fecha y Hora
df['Timestamp'] = pd.to_datetime(df['Fecha_dt'].astype(str) + ' ' + df['Hora_str'], errors='coerce')

display(df[['Fecha', 'Intervalo', 'Timestamp', '% SLA', 'AHT (Min)']].head())


,Fecha,Intervalo,Timestamp,% SLA,AHT (Min)
0,2024-11-25 00:00:00,1899-12-30 00:00:00,2024-11-25 00:00:00,0.8802,6.793165
1,2024-11-25 00:00:00,1899-12-30 00:30:00,2024-11-25 00:30:00,0.8056,5.348102
2,2024-11-25 00:00:00,1899-12-30 01:00:00,2024-11-25 01:00:00,0.7863,5.451179
3,2024-11-25 00:00:00,1899-12-30 01:30:00,2024-11-25 01:30:00,0.8889,6.053495
4,2024-11-25 00:00:00,1899-12-30 02:00:00,2024-11-25 02:00:00,0.9565,8.575402


## Feature Engineering Temporal
Derivacion de dimensiones de tiempo (`hora_del_dia` y `dia_de_la_semana`) a partir del Timestamp base.
Estas variables son el input critico para la segmentacion horaria y el posterior modelado de forecasting (Modulo 3).

In [3]:
df['hora_del_dia'] = df['Timestamp'].dt.hour
df['dia_de_la_semana'] = df['Timestamp'].dt.day_name()

display(df[['Timestamp', 'hora_del_dia', 'dia_de_la_semana']].head())


,Timestamp,hora_del_dia,dia_de_la_semana
0,2024-11-25 00:00:00,0,Monday
1,2024-11-25 00:30:00,0,Monday
2,2024-11-25 01:00:00,1,Monday
3,2024-11-25 01:30:00,1,Monday
4,2024-11-25 02:00:00,2,Monday


## Exploratory Data Analysis (EDA)
Analisis exploratorio sobre las principales metricas de eficiencia operativa. Evaluacion interactiva
del Service Level Agreement (SLA), Average Handle Time (AHT) y el impacto del Contact Rate en el Customer Satisfaction (CSAT).

In [4]:
# 1. Evolución temporal del '% SLA'
df_sla = df.dropna(subset=['Timestamp', '% SLA']).sort_values('Timestamp')
fig_sla = px.line(df_sla, x='Timestamp', y='% SLA', title="Evolución Temporal del % SLA (Service Level Agreement)")
fig_sla.add_hline(y=0.90, line_dash="dash", line_color="red", annotation_text="Meta (90%)")
fig_sla.show()

# 2. Evolución temporal del 'AHT (Min)'
df_aht = df.dropna(subset=['Timestamp', 'AHT (Min)']).sort_values('Timestamp')
fig_aht = px.line(df_aht, x='Timestamp', y='AHT (Min)', title="Evolución Temporal del AHT (Average Handle Time)")
fig_aht.add_hline(y=6.0, line_dash="dash", line_color="green", annotation_text="Meta (6 min)")
fig_aht.show()

# 3. Scatter plot de 'Csat_Gral' vs 'Contact Rate'
df_scatter = df.dropna(subset=['Csat_Gral', 'Contact Rate'])
fig_scatter = px.scatter(df_scatter, x='Contact Rate', y='Csat_Gral',
                         color='hora_del_dia',
                         title="Relación entre Contact Rate y Satisfacción General (CSAT)",
                         labels={'Contact Rate': 'Contact Rate (Volumen/Tasa)', 'Csat_Gral': 'CSAT Promedio'},
                         trendline="ols")
fig_scatter.show()


## Agregacion de Resultados
Agrupacion de la data transaccional a nivel horario para facilitar el computo de metricas agregadas y su persistencia.

In [5]:
# Agrupación por hora para extraer promedios y sumas
df_resumen = df.groupby('hora_del_dia').agg(
    SLA_Promedio=('% SLA', 'mean'),
    AHT_Promedio=('AHT (Min)', 'mean'),
    CSAT_Promedio=('Csat_Gral', 'mean'),
    Inflow_Total=('Inflow', 'sum')
).reset_index()

# Calculamos un score de 'Peor Desempeño' combinando el ranking de SLA (ascendente, los más bajos primero)
# y AHT (descendente, los más altos primero)
df_resumen['Rank_SLA'] = df_resumen['SLA_Promedio'].rank(ascending=True)
df_resumen['Rank_AHT'] = df_resumen['AHT_Promedio'].rank(ascending=False)
df_resumen['Score_Friccion'] = df_resumen['Rank_SLA'] + df_resumen['Rank_AHT']

peores_5_horas = df_resumen.sort_values('Score_Friccion').head(5).drop(columns=['Rank_SLA', 'Rank_AHT', 'Score_Friccion'])

print("Top 5 peores horas operativas (Menor SLA, Mayor AHT):")
display(peores_5_horas)


Top 5 peores horas operativas (Menor SLA, Mayor AHT):


,hora_del_dia,SLA_Promedio,AHT_Promedio,CSAT_Promedio,Inflow_Total
19,19,0.750939,7.209042,2.493251,14834
17,17,0.694596,6.798613,2.476624,12068
18,18,0.769436,7.339815,2.514618,12304
4,4,0.834704,8.524268,2.385246,612
21,21,0.674982,6.682605,2.498721,15149


## Persistencia — Artefacto de Salida
Exportacion del dataset enriquecido a `data/modulo1_resultados.csv`.
Este artefacto es consumido directamente por los componentes predictivos y la capa de visualizacion del Dashboard (Modulo 5).

In [6]:
output_file = 'data/modulo1_resultados.csv'
peores_5_horas.to_csv(output_file, index=False)
print(f"Resultados agregados exportados con éxito a: {output_file}")


Resultados agregados exportados con éxito a: data/modulo1_resultados.csv


## Estrategias Propuestas para Mejorar CSAT, SLA y AHT

1. **Estrategia SLA (Balanceo de Capacidad):** Existe un deficit severo de cobertura durante las horas pico de volumen (Inflow). Se recomienda reestructurar la asignacion de turnos (shift bidding), desplazando Headcount de horas valle hacia estas franjas para estabilizar el SLA por encima de la meta del 90%.
2. **Estrategia AHT (Mitigacion de Embotellamiento):** Durante los picos de saturacion, el Average Handle Time (AHT) se incrementa de forma no lineal. Esto indica que los agentes enfrentan mayor presion cognitiva. Implementar automatizacion (triaje, deflection via bots) en estas franjas mitigara el volumen derivado a operadores humanos.
3. **Estrategia CSAT (Correlacion NLP):** Al estabilizar el SLA y el AHT, se reduce el tiempo de espera y resolucion, lo cual es la principal causa subyacente de las reseñas negativas detectadas en el modelo NLP. Atacar la ineficiencia temporal mejorara el CSAT de forma indirecta pero contundente.